# JobFit AI: Short Agents SDK Notebook

This version keeps the notebook intentionally small: one Kimi 2.6 agent, two web tools, and one run cell. It uses the OpenAI Agents SDK with Moonshot/Kimi's OpenAI-compatible Chat Completions endpoint.

In [1]:
import json
import os

import requests
from agents import Agent, AsyncOpenAI, ModelSettings, OpenAIChatCompletionsModel, RunConfig, Runner, function_tool, set_tracing_disabled
from IPython.display import Markdown, display
from pypdf import PdfReader

KIMI_MODEL = "kimi-k2.6"
KIMI_BASE_URL = "https://api.moonshot.ai/v1"
OLOSTEP_SEARCH_URL = "https://api.olostep.com/v1/searches"
OLOSTEP_SCRAPE_URL = "https://api.olostep.com/v1/scrapes"
MAX_AGENT_TURNS = 25

cv_path = "abid-resume.pdf"
preferences = """
Remote data science, AI writer, or technical writer roles in AI, machine learning, data science, or cloud.
Prefer technical content, tutorials, developer education, research writing, and AI product storytelling.
""".strip()

set_tracing_disabled(True)

kimi_client = AsyncOpenAI(
    api_key=os.environ["MOONSHOT_API_KEY"],
    base_url=KIMI_BASE_URL,
)

kimi_model = OpenAIChatCompletionsModel(
    model=KIMI_MODEL,
    openai_client=kimi_client,
)

model_settings = ModelSettings(
    tool_choice="auto",
    parallel_tool_calls=True,
    extra_body={"thinking": {"type": "disabled"}},
)

run_config = RunConfig(
    workflow_name="JobFit AI Kimi Search",
    tracing_disabled=True,
)


## Read The CV

In [2]:
reader = PdfReader(cv_path)
cv_text = "\n".join(page.extract_text() or "" for page in reader.pages)[:12000]
print(f"Loaded {len(cv_text):,} characters from {cv_path}")

Loaded 2,946 characters from abid-resume.pdf


## Prompts


In [3]:
AGENT_INSTRUCTIONS = """
You are JobFit AI, a focused job-search agent.

Tool plan:
- Call search_jobs exactly once with limit 8.
- Read at most 3 direct job pages with read_job_page.
- After reading up to 3 pages, stop using tools and write the report.
- Search again only if the first search returns zero usable jobs.
- Avoid broad search pages, expired jobs, and LinkedIn unless no better source exists.

Report rules:
- Keep the report simple, clear, and practical.
- Use short bullets.
- Do not use em dashes.
- Do not use contractions.
- Do not add text before or after the report.
- End after the final Job Notes entry.
- Include at least 5 ranked jobs if the search results contain at least 5 usable jobs.
- If only 3 pages were scraped, use backup jobs from search results when they look usable.
- Every job must include a clickable Markdown link.
- Every job must have one apply decision: Apply, Maybe, or Do not apply.

Use exactly this Markdown structure:

# JobFit AI Report

## Best Match

- **Role:** <job title>
- **Company:** <company>
- **Apply decision:** Apply / Maybe / Do not apply
- **Fit score:** <score>/100
- **Link:** [Apply here](<job url>)

**Why this is the best match:**

- <specific reason>
- <specific reason>
- <specific reason>

## Ranked Jobs

| Rank | Role | Company | Apply? | Fit | Link |
| --- | --- | --- | --- | --- | --- |
| 1 | <role> | <company> | Apply / Maybe / Do not apply | <score>/100 | [Apply here](<url>) |

## Job Notes

### 1. <Role> at <Company>

- **Apply decision:** Apply / Maybe / Do not apply
- **Fit score:** <score>/100
- **Link:** [Apply here](<job url>)

**Why it fits:**

- <bullet>
- <bullet>

**Concerns:**

- <bullet>
- <bullet>

**Application angle:**

- <how the person should position their CV/application>
""".strip()

RUN_PROMPT_TEMPLATE = """
Find current job postings for this candidate and rank them by fit.

Keep the run simple:
- one search
- up to three page reads
- final report

The final report must follow AGENT_INSTRUCTIONS exactly.
Use simple wording. Do not use em dashes. Do not use contractions.

Candidate CV:
{cv_text}

Preferences:
{preferences}
""".strip()


## Give The Agent Web Access

In [4]:
@function_tool
def search_jobs(query: str, limit: int = 8) -> str:
    """Search the web for job listings and return compact JSON results."""
    response = requests.post(
        OLOSTEP_SEARCH_URL,
        headers={"Authorization": f"Bearer {os.environ['OLOSTEP_API_KEY']}", "Content-Type": "application/json"},
        json={"query": query},
        timeout=60,
    )
    response.raise_for_status()
    links = response.json().get("result", {}).get("links", [])[:limit]
    results = [
        {"title": item.get("title", "Untitled"), "url": item.get("url"), "description": item.get("description", "")}
        for item in links
        if isinstance(item, dict) and item.get("url")
    ]
    return json.dumps(results, ensure_ascii=False)


@function_tool
def read_job_page(url: str) -> str:
    """Scrape one job listing URL and return markdown text."""
    response = requests.post(
        OLOSTEP_SCRAPE_URL,
        headers={"Authorization": f"Bearer {os.environ['OLOSTEP_API_KEY']}", "Content-Type": "application/json"},
        json={"url_to_scrape": url, "formats": ["markdown"]},
        timeout=120,
    )
    response.raise_for_status()
    markdown = response.json().get("result", {}).get("markdown_content") or ""
    return markdown[:8000]

agent = Agent(
    name="JobFit AI",
    model=kimi_model,
    model_settings=model_settings,
    tools=[search_jobs, read_job_page],
    instructions=AGENT_INSTRUCTIONS,

)


## Run JobFit

In [5]:
prompt = RUN_PROMPT_TEMPLATE.format(cv_text=cv_text, preferences=preferences)


print("Starting agent run")

result = Runner.run_streamed(
    agent,
    prompt,
    max_turns=MAX_AGENT_TURNS,
    run_config=run_config,
)

async for event in result.stream_events():
    if event.type == "agent_updated_stream_event":
        print(f"Agent: {event.new_agent.name}")
    elif event.type == "run_item_stream_event":
        item = event.item
        if event.name == "tool_called":
            raw = item.raw_item
            tool_name = raw.get("name") if isinstance(raw, dict) else getattr(raw, "name", "tool")
            arguments = raw.get("arguments") if isinstance(raw, dict) else getattr(raw, "arguments", "")
            arguments = str(arguments).replace(chr(10), " ")[:500]
            print(f"Tool call: {tool_name}")
            if arguments:
                print(f"Parameters: {arguments}")
        elif event.name == "tool_output":
            print(f"Tool output: {len(str(item.output)):,} chars")
        elif event.name == "message_output_created":
            print("Final message ready")

report = result.final_output
globals()["report"] = report

print("Run complete")
print(f"Model responses: {len(result.raw_responses)}")
print(f"Run items: {len(result.new_items)}")
print(f"Final output: {len(str(report)):,} chars")


Starting agent run
Agent: JobFit AI
Tool call: search_jobs
Parameters: {"query":"remote data science writer technical writer AI machine learning content editor","limit":8}
Final message ready
Tool output: 2,445 chars
Tool call: read_job_page
Parameters: {"url":"https://www.indeed.com/q-data-science-writer-jobs.html"}
Final message ready
Tool output: 8,000 chars
Tool call: read_job_page
Parameters: {"url":"https://www.builtinnyc.com/jobs/remote/data-analytics/data-science"}
Final message ready
Tool output: 8,000 chars
Tool call: read_job_page
Parameters: {"url":"https://www.virtualvocations.com/jobs/q-data+scientist+remote+jobs/c-writing/d-336"}
Final message ready
Tool output: 5,075 chars
Final message ready
Run complete
Model responses: 5
Run items: 13
Final output: 5,931 chars


In [6]:
display(Markdown(report))

# JobFit AI Report

## Best Match

- **Role:** Data Scientist
- **Company:** Pluralsight
- **Apply decision:** Maybe
- **Fit score:** 62/100
- **Link:** [Apply here](https://www.builtinnyc.com/job/data-scientist/8853432)

**Why this is the best match:**

- Remote or hybrid option fits candidate preference for remote work
- Edtech company aligns with candidate experience at DataCamp and KDnuggets
- Role involves communicating data-driven insights, which matches the candidate content and editorial background

## Ranked Jobs

| Rank | Role | Company | Apply? | Fit | Link |
| --- | --- | --- | --- | --- | --- |
| 1 | Data Scientist | Pluralsight | Maybe | 62/100 | [Apply here](https://www.builtinnyc.com/job/data-scientist/8853432) |
| 2 | Data Scientist | Dropbox | Maybe | 55/100 | [Apply here](https://www.builtinnyc.com/job/data-scientist/8917873) |
| 3 | Data Scientist, Product Analytics | Whatnot | Do not apply | 48/100 | [Apply here](https://www.builtinnyc.com/job/data-scientist-product-analytics/7957130) |
| 4 | Graduate Research Co-op, Population Genetics/Genomic Data Science | Ancestry.com | Maybe | 45/100 | [Apply here](https://www.indeed.com/rc/clk?jk=3e9d249194d3ef64&bb=dnMi8mJNDT0hfoLdrgxgHCvdCO5k1Y_Yblpl7G-9lbdGq3Aoe-osoeldk9v6zyb75qGDwpCGJPvPuo3TCb9Yidqmi7_fs5w0v9PAWyXzowoAR5iMyuEK9jpZ2PJdJt00&xkcb=SoCj67M3k6R86ZSgJB0KbzkdCdPP&fccid=8a644f7a25dca5dc&vjs=3) |
| 5 | Technical Consultant | Virtual Vocations listing | Do not apply | 35/100 | [Apply here](https://www.virtualvocations.com/job/technical-consultant-3020433-i.html) |

## Job Notes

### 1. Data Scientist at Pluralsight

- **Apply decision:** Maybe
- **Fit score:** 62/100
- **Link:** [Apply here](https://www.builtinnyc.com/job/data-scientist/8853432)

**Why it fits:**

- Edtech environment similar to DataCamp and KDnuggets experience
- Remote or hybrid work option matches candidate preference
- Requires communicating data insights, which aligns with writing and editorial skills

**Concerns:**

- Role focuses on algorithm development rather than content creation
- May require more hands-on coding and less writing than candidate prefers
- Salary range of 122K-160K may be lower than candidate expectations given experience

**Application angle:**

- Emphasize experience translating complex data science concepts into accessible content
- Highlight DataCamp and KDnuggets editorial work as evidence of communication skills
- Showcase ML/DL project portfolio to demonstrate technical depth

### 2. Data Scientist at Dropbox

- **Apply decision:** Maybe
- **Fit score:** 55/100
- **Link:** [Apply here](https://www.builtinnyc.com/job/data-scientist/8917873)

**Why it fits:**

- Remote work option available
- Involves analyzing user behavior and business metrics
- Cloud and software company aligns with candidate technology interests

**Concerns:**

- Heavy focus on SaaS growth metrics and experimentation, not content
- Requires Hadoop and R skills which are not highlighted in candidate CV
- More product analytics than technical writing or education

**Application angle:**

- Frame editorial experience as user behavior analysis through content engagement metrics
- Highlight SQL and data analysis skills from portfolio projects
- Mention experience with data-driven storytelling for technical audiences

### 3. Data Scientist, Product Analytics at Whatnot

- **Apply decision:** Do not apply
- **Fit score:** 48/100
- **Link:** [Apply here](https://www.builtinnyc.com/job/data-scientist-product-analytics/7957130)

**Why it fits:**

- Remote option available
- Data science role in tech company

**Concerns:**

- Ecommerce and mobile focus does not match candidate AI/ML content expertise
- Heavy emphasis on product analytics tools like BigQuery, dbt, Redshift, Snowflake, Spark
- No connection to technical writing, education, or AI storytelling
- In-office or remote hybrid may not meet fully remote preference

**Application angle:**

- Not recommended unless candidate wants to pivot to ecommerce analytics
- If applying, focus on data analysis project experience and SQL skills

### 4. Graduate Research Co-op, Population Genetics/Genomic Data Science at Ancestry.com

- **Apply decision:** Maybe
- **Fit score:** 45/100
- **Link:** [Apply here](https://www.indeed.com/rc/clk?jk=3e9d249194d3ef64&bb=dnMi8mJNDT0hfoLdrgxgHCvdCO5k1Y_Yblpl7G-9lbdGq3Aoe-osoeldk9v6zyb75qGDwpCGJPvPuo3TCb9Yidqmi7_fs5w0v9PAWyXzowoAR5iMyuEK9jpZ2PJdJt00&xkcb=SoCj67M3k6R86ZSgJB0KbzkdCdPP&fccid=8a644f7a25dca5dc&vjs=3)

**Why it fits:**

- Remote work option
- Involves Python and machine learning for data analysis
- Research-oriented role with scientific writing potential

**Concerns:**

- Part-time co-op position, not full-time
- Population genetics focus is niche and not aligned with candidate AI/ML content expertise
- Graduate level position may not match seniority of candidate with 4+ years experience

**Application angle:**

- Highlight Python and ML skills from portfolio projects
- Emphasize research writing experience from freelance and editorial work
- Position as experienced professional bringing content expertise to research team

### 5. Technical Consultant at Virtual Vocations listing

- **Apply decision:** Do not apply
- **Fit score:** 35/100
- **Link:** [Apply here](https://www.virtualvocations.com/job/technical-consultant-3020433-i.html)

**Why it fits:**

- Remote work option
- Technical role involving data science background

**Concerns:**

- Client success and solution engineering focus, not writing or content
- Requires 3+ years in technical consulting or solution engineering
- No alignment with editorial, copywriting, or technical writing skills
- Skills listed (Python, NumPy, Pandas, NLTK) suggest engineering role, not content creation

**Application angle:**

- Not recommended for this candidate profile
- If applying, focus on technical project delivery and client communication from freelance work

## Save Markdown Report


In [7]:
output_path = "jobfit_report.md"
with open(output_path, "w", encoding="utf-8") as file:
    file.write(report)

print(f"Saved {output_path}")

Saved jobfit_report.md
